
## 1) sorteio Mega Sena, estudo por maiores frequências de números por meses]

## 2) Forecast utilizando o prophet


anaconda ambiente orange3



In [1]:
import pandas as pd

dados retirados daqui https://loterias.caixa.gov.br/Paginas/Mega-Sena.aspx

In [2]:
nome_do_arquivo = "Mega-Sena.xlsx"
df_volume = pd.read_excel(nome_do_arquivo, sheet_name='MEGA SENA')
print("Dados da aba Volume:")
print(df_volume.head())

Dados da aba Volume:
   Concurso Data do Sorteio  Bola1  Bola2  Bola3  Bola4  Bola5  Bola6  \
0         1      11/03/1996      4      5     30     33     41     52   
1         2      18/03/1996      9     37     39     41     43     49   
2         3      25/03/1996     10     11     29     30     36     47   
3         4      01/04/1996      1      5      6     27     42     59   
4         5      08/04/1996      1      2      6     16     19     46   

   Ganhadores 6 acertos Cidade / UF Rateio 6 acertos  Ganhadores 5 acertos  \
0                     0         NaN           R$0,00                    17   
1                     1          PR   R$2.307.162,23                    65   
2                     2      RN; SP     R$391.192,51                    62   
3                     0         NaN           R$0,00                    39   
4                     0         NaN           R$0,00                    98   

  Rateio 5 acertos  Ganhadores 4 acertos Rateio 4 acertos Acumulado 6 a

In [3]:
df_volume.columns

Index(['Concurso', 'Data do Sorteio', 'Bola1', 'Bola2', 'Bola3', 'Bola4',
       'Bola5', 'Bola6', 'Ganhadores 6 acertos', 'Cidade / UF',
       'Rateio 6 acertos', 'Ganhadores 5 acertos', 'Rateio 5 acertos',
       'Ganhadores 4 acertos', 'Rateio 4 acertos', 'Acumulado 6 acertos',
       'Arrecadação Total', 'Estimativa prêmio',
       'Acumulado Sorteio Especial Mega da Virada', 'Observação'],
      dtype='object')

In [4]:
df_pesquisa =  df_volume[['Data do Sorteio', 'Bola1', 'Bola2', 'Bola3', 'Bola4',
       'Bola5', 'Bola6']]

In [5]:
df_pesquisa.tail()

,Data do Sorteio,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
2945,02/12/2025,4,13,17,21,49,54
2946,04/12/2025,4,10,15,37,39,44
2947,06/12/2025,6,24,37,52,53,58
2948,09/12/2025,4,6,11,38,49,54
2949,11/12/2025,21,23,42,49,50,60


In [6]:
df_pesquisa.info()

print(df_pesquisa.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2950 entries, 0 to 2949
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Data do Sorteio  2950 non-null   object
 1   Bola1            2950 non-null   int64 
 2   Bola2            2950 non-null   int64 
 3   Bola3            2950 non-null   int64 
 4   Bola4            2950 non-null   int64 
 5   Bola5            2950 non-null   int64 
 6   Bola6            2950 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 161.5+ KB
Data do Sorteio    0
Bola1              0
Bola2              0
Bola3              0
Bola4              0
Bola5              0
Bola6              0
dtype: int64


In [7]:
df_pesquisa['Data do Sorteio'] = pd.to_datetime(
    df_pesquisa['Data do Sorteio'],
    format='%d/%m/%Y'  # Informa ao Pandas que o formato é Dia/Mês/Ano
)

/tmp/ipykernel_21950/1612541821.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pesquisa['Data do Sorteio'] = pd.to_datetime(


In [8]:
df_pesquisa.tail()

,Data do Sorteio,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
2945,2025-12-02,4,13,17,21,49,54
2946,2025-12-04,4,10,15,37,39,44
2947,2025-12-06,6,24,37,52,53,58
2948,2025-12-09,4,6,11,38,49,54
2949,2025-12-11,21,23,42,49,50,60


In [9]:
# 1. Preparação: Criar uma coluna 'Mês'
df_pesquisa['Mês'] = df_pesquisa['Data do Sorteio'].dt.month

/tmp/ipykernel_21950/1195303281.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pesquisa['Mês'] = df_pesquisa['Data do Sorteio'].dt.month


In [10]:
# 1. Preparação: Criar uma coluna 'Mês'
df_pesquisa['Mês'] = df_pesquisa['Data do Sorteio'].dt.month

# 2. Consolidação: Empilhar todas as colunas de bolas em uma única coluna
# Cria uma lista com o nome das colunas de interesse (Bola1 a Bola6)
colunas_bolas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']

# Usa pd.melt para "despivotar" as colunas, empilhando-as
df_longo = pd.melt(
    df_pesquisa,
    id_vars=['Mês'],
    value_vars=colunas_bolas,
    value_name='Número Sorteado'
)
# 

# 3. Agrupamento e Contagem: Calcular a frequência de cada número por Mês
frequencia_mensal = df_longo.groupby(['Mês', 'Número Sorteado']).size().reset_index(name='Frequência')

# 4. Seleção do Top 1: Encontrar o número mais frequente em cada Mês
# Transforma o DataFrame em uma Series onde o índice é 'Mês' e o valor é 'Frequência'
# E mantém apenas a linha com a maior frequência para cada mês
resultado_final_por_Mes = frequencia_mensal.loc[frequencia_mensal.groupby('Mês')['Frequência'].idxmax()]

# 5. Formatação (Opcional): Organizar e nomear o Mês
# Mapeia o número do mês para o nome do mês para melhor visualização
nomes_meses = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}
resultado_final_por_Mes['Mês'] = resultado_final_por_Mes['Mês'].map(nomes_meses)

print(resultado_final_por_Mes.sort_values(by='Mês', key=lambda x: x.map({v: k for k, v in nomes_meses.items()})))

           Mês  Número Sorteado  Frequência
43     Janeiro               44          35
70   Fevereiro               11          36
173      Março               54          36
209      Abril               30          36
262       Maio               23          36
315      Junho               16          36
386      Julho               27          36
454     Agosto               35          40
508   Setembro               29          38
550    Outubro               11          38
609   Novembro               10          37
691   Dezembro               32          36


/tmp/ipykernel_21950/3273983781.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pesquisa['Mês'] = df_pesquisa['Data do Sorteio'].dt.month


In [11]:
print('maior número sorteado por Mês')
resultado_final_por_Mes

maior número sorteado por Mês


,Mês,Número Sorteado,Frequência
43,Janeiro,44,35
70,Fevereiro,11,36
173,Março,54,36
209,Abril,30,36
262,Maio,23,36
315,Junho,16,36
386,Julho,27,36
454,Agosto,35,40
508,Setembro,29,38
550,Outubro,11,38


In [12]:
# 1. Preparação: Criar as colunas 'Ano' e 'Mês'
df_pesquisa['Ano'] = df_pesquisa['Data do Sorteio'].dt.year
df_pesquisa['Mês'] = df_pesquisa['Data do Sorteio'].dt.month
df_pesquisa['Mês-Ano'] = df_pesquisa['Data do Sorteio'].dt.to_period('M')

# Lista das colunas que queremos analisar
colunas_bolas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']
print('frequência do sorteio em cada mês em em cada bola')
# Dicionário para armazenar o resultado final de cada bola
resultado_por_bola = {}

for bola in colunas_bolas:
    # 2. Agrupamento e Contagem: Conta a frequência de cada número
    # O agrupamento é feito por Mês-Ano e pelo valor da Bola (o número sorteado)
    frequencia_tripla = df_pesquisa.groupby(['Mês-Ano', bola]).size().reset_index(name='Frequência')

    # 3. Seleção do Top 1: Encontra o número mais frequente em cada Mês-Ano
    # Para cada Mês-Ano, encontramos o índice (linha) da maior Frequência
    idx_max = frequencia_tripla.groupby('Mês-Ano')['Frequência'].idxmax()

    # Filtramos o DataFrame para manter apenas as linhas com as maiores frequências
    top_numero_por_mes_ano = frequencia_tripla.loc[idx_max].copy()

    # Renomeia a coluna da bola para algo mais descritivo
    top_numero_por_mes_ano.rename(columns={bola: 'Número Mais Frequente'}, inplace=True)
    
    # Armazena o resultado no dicionário
    resultado_por_bola[bola] = top_numero_por_mes_ano.sort_values(by='Mês-Ano')

    # Opcional: Imprime o resultado para a bola atual
    print(f"--- Resultados para {bola} ---")
    print(top_numero_por_mes_ano[['Mês-Ano', 'Número Mais Frequente', 'Frequência']].head())
    print(top_numero_por_mes_ano[['Mês-Ano', 'Número Mais Frequente', 'Frequência']].tail(12))
    
    print("\n")

# 4. Exibição do Resultado (Opção de concatenação ou inspeção individual)
# Se você quiser ver todos os resultados de uma vez, pode concatenar:
# resultado_final_completo = pd.concat(resultado_por_bola, axis=0)
# print("--- Tabela Final Completa ---")
# print(resultado_final_completo)

print("A análise está concluída. Os resultados estão armazenados no dicionário 'resultado_por_bola'.")

frequência do sorteio em cada mês em em cada bola
--- Resultados para Bola1 ---
    Mês-Ano  Número Mais Frequente  Frequência
0   1996-03                      4           1
3   1996-04                      1           2
7   1996-05                      4           2
10  1996-06                      2           1
15  1996-07                      6           2
      Mês-Ano  Número Mais Frequente  Frequência
2281  2025-01                     11           2
2286  2025-02                      1           3
2294  2025-03                      1           4
2302  2025-04                      2           2
2310  2025-05                      2           5
2317  2025-06                      4           3
2326  2025-07                      5           2
2335  2025-08                      2           2
2344  2025-09                      3           2
2356  2025-10                      4           3
2366  2025-11                      8           2
2372  2025-12                      4           3



/tmp/ipykernel_21950/1975417349.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pesquisa['Ano'] = df_pesquisa['Data do Sorteio'].dt.year
/tmp/ipykernel_21950/1975417349.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pesquisa['Mês'] = df_pesquisa['Data do Sorteio'].dt.month
/tmp/ipykernel_21950/1975417349.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

In [13]:
print('frequência do sorteio do mês de uma forma geral em em cada bola')


# 1. Preparação: Criar as colunas 'Ano' e 'Mês'
#df_pesquisa['Ano'] = df_pesquisa['Data do Sorteio'].dt.year
df_pesquisa['Mês'] = df_pesquisa['Data do Sorteio'].dt.month
#df_pesquisa['Mês-Ano'] = df_pesquisa['Data do Sorteio'].dt.to_period('M')

# Lista das colunas que queremos analisar
colunas_bolas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']

# Dicionário para armazenar o resultado final de cada bola
resultado_por_bola = {}

for bola in colunas_bolas:
    # 2. Agrupamento e Contagem: Conta a frequência de cada número
    # O agrupamento é feito por Mês-Ano e pelo valor da Bola (o número sorteado)
    frequencia_tripla = df_pesquisa.groupby(['Mês', bola]).size().reset_index(name='Frequência')

    # 3. Seleção do Top 1: Encontra o número mais frequente em cada Mês-Ano
    # Para cada Mês-Ano, encontramos o índice (linha) da maior Frequência
    idx_max = frequencia_tripla.groupby('Mês')['Frequência'].idxmax()

    # Filtramos o DataFrame para manter apenas as linhas com as maiores frequências
    top_numero_por_mes_ano = frequencia_tripla.loc[idx_max].copy()

    # Renomeia a coluna da bola para algo mais descritivo
    top_numero_por_mes_ano.rename(columns={bola: 'Número Mais Frequente'}, inplace=True)
    
    # Armazena o resultado no dicionário
    resultado_por_bola[bola] = top_numero_por_mes_ano.sort_values(by='Mês')

    # Opcional: Imprime o resultado para a bola atual
    print(f"--- Resultados para {bola} ---")
    display(top_numero_por_mes_ano[['Mês', 'Número Mais Frequente', 'Frequência']])
    
    print("\n")

# 4. Exibição do Resultado (Opção de concatenação ou inspeção individual)
# Se você quiser ver todos os resultados de uma vez, pode concatenar:
# resultado_final_completo = pd.concat(resultado_por_bola, axis=0)
# print("--- Tabela Final Completa ---")
# print(resultado_final_completo)

print("A análise está concluída. Os resultados estão armazenados no dicionário 'resultado_por_bola'.")

frequência do sorteio do mês de uma forma geral em em cada bola
--- Resultados para Bola1 ---


/tmp/ipykernel_21950/765663788.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pesquisa['Mês'] = df_pesquisa['Data do Sorteio'].dt.month


,Mês,Número Mais Frequente,Frequência
0,1,1,29
30,2,1,25
61,3,4,27
92,4,4,25
119,5,1,28
148,6,1,22
181,7,4,27
211,8,1,27
243,9,1,23
275,10,3,29




--- Resultados para Bola2 ---


,Mês,Número Mais Frequente,Frequência
11,1,13,15
48,2,10,14
89,3,13,14
136,4,17,15
168,5,12,15
208,6,16,16
244,7,10,16
283,8,9,16
330,9,13,18
365,10,11,24


...



--- Resultados para Bola5 ---


,Mês,Número Mais Frequente,Frequência
21,1,42,15
69,2,51,16
105,3,49,15
143,4,49,15
186,5,50,16
218,6,42,16
257,7,43,14
294,8,41,16
329,9,41,19
373,10,46,16




--- Resultados para Bola6 ---


,Mês,Número Mais Frequente,Frequência
26,1,58,24
53,2,59,23
84,3,60,25
110,4,59,25
144,5,60,27
175,6,60,21
207,7,59,29
240,8,59,26
271,9,60,32
300,10,60,27




A análise está concluída. Os resultados estão armazenados no dicionário 'resultado_por_bola'.


In [14]:
from datetime import datetime, timedelta

# 1. Obter a data atual do sistema
data_atual = datetime.now()

# dias desejados
dias_desejados = 45
# 2. Calcular a data de corte (45 dias atrás)
# O timedelta subtrai 45 dias da data atual
data_corte = data_atual - timedelta(days=dias_desejados)

# 3. Filtrar o DataFrame
# Seleciona todas as linhas onde a coluna 'Data' é maior ou igual à data de corte
df_ultimos_selecionados_dias = df_pesquisa[df_pesquisa['Data do Sorteio'] >= data_corte].reset_index(drop=True)

# Opcional: Mostrar a data de corte e o número de linhas filtradas
print(f"Data Atual: {data_atual.strftime('%Y-%m-%d')}")
print(f"Data de Corte ({dias_desejados} dias atrás): {data_corte.strftime('%Y-%m-%d')}")
print(f"DataFrame original tinha {len(df_pesquisa)} linhas.")
print(f"DataFrame filtrado tem {len(df_ultimos_selecionados_dias)} linhas.")

Data Atual: 2025-12-13
Data de Corte (45 dias atrás): 2025-10-29
DataFrame original tinha 2950 linhas.
DataFrame filtrado tem 17 linhas.


In [15]:
def funcao_por_mes(df_analise):

    
    # 1. Preparação: Criar as colunas 'Ano' e 'Mês'
    #df_analise['Ano'] = df_analise['Data do Sorteio'].dt.year
    df_analise['Mês'] = df_analise['Data do Sorteio'].dt.month
    #df_analise['Mês-Ano'] = df_analise['Data do Sorteio'].dt.to_period('M')
    
    # Lista das colunas que queremos analisar
    colunas_bolas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']
    
    # Dicionário para armazenar o resultado final de cada bola
    resultado_por_bola = {}
    
    for bola in colunas_bolas:
        # 2. Agrupamento e Contagem: Conta a frequência de cada número
        # O agrupamento é feito por Mês-Ano e pelo valor da Bola (o número sorteado)
        frequencia_tripla = df_analise.groupby(['Mês', bola]).size().reset_index(name='Frequência')
    
        # 3. Seleção do Top 1: Encontra o número mais frequente em cada Mês-Ano
        # Para cada Mês-Ano, encontramos o índice (linha) da maior Frequência
        idx_max = frequencia_tripla.groupby('Mês')['Frequência'].idxmax()
    
        # Filtramos o DataFrame para manter apenas as linhas com as maiores frequências
        top_numero_por_mes_ano = frequencia_tripla.loc[idx_max].copy()
    
        # Renomeia a coluna da bola para algo mais descritivo
        top_numero_por_mes_ano.rename(columns={bola: 'Número Mais Frequente'}, inplace=True)
        
        # Armazena o resultado no dicionário
        resultado_por_bola[bola] = top_numero_por_mes_ano.sort_values(by='Mês')
    
        # Opcional: Imprime o resultado para a bola atual
        print(f"--- Resultados para {bola} ---")
        display(top_numero_por_mes_ano[['Mês', 'Número Mais Frequente', 'Frequência']])
        
        print("\n")
    
    # 4. Exibição do Resultado (Opção de concatenação ou inspeção individual)
    # Se você quiser ver todos os resultados de uma vez, pode concatenar:
    # resultado_final_completo = pd.concat(resultado_por_bola, axis=0)
    # print("--- Tabela Final Completa ---")
    # print(resultado_final_completo)
    
    print("A análise está concluída. Os resultados estão armazenados no dicionário 'resultado_por_bola'.")
    #return top_numero_por_mes_ano[['Mês', 'Número Mais Frequente', 'Frequência']]

In [16]:
df_resultado = funcao_por_mes(df_ultimos_selecionados_dias)

--- Resultados para Bola1 ---


,Mês,Número Mais Frequente,Frequência
0,10,9,1
4,11,8,2
10,12,4,3




--- Resultados para Bola2 ---


,Mês,Número Mais Frequente,Frequência
0,10,17,1
9,11,30,2
11,12,6,1




--- Resultados para Bola3 ---


...



--- Resultados para Bola5 ---


,Mês,Número Mais Frequente,Frequência
0,10,33,1
8,11,44,2
12,12,49,2




--- Resultados para Bola6 ---


,Mês,Número Mais Frequente,Frequência
0,10,59,1
10,11,60,2
12,12,54,2




A análise está concluída. Os resultados estão armazenados no dicionário 'resultado_por_bola'.


In [17]:
def Funcao_forma_quinzenal(df_analise):
    
    # É fundamental garantir que a coluna 'Data do Sorteio' seja do tipo datetime
    if not pd.api.types.is_datetime64_any_dtype(df_analise['Data do Sorteio']):
        print("A coluna 'Data do Sorteio' não é datetime. Tentando conversão...")
        try:
            # Tenta a conversão, assumindo o formato brasileiro D/M/A
            df_analise['Data do Sorteio'] = pd.to_datetime(df_analise['Data do Sorteio'], dayfirst=True)
        except Exception as e:
            print(f"Erro ao converter a data: {e}")
            return
    
    # 1. Preparação: Criar colunas de Agrupamento
    df_analise['Mês'] = df_analise['Data do Sorteio'].dt.month
    
    # NOVO PASSO CRÍTICO: Criar a coluna 'Quinzena'
    # Se o dia do mês for <= 15, é a 1ª quinzena. Caso contrário, é a 2ª.
    df_analise['Quinzena'] = df_analise['Data do Sorteio'].dt.day.apply(lambda x: 1 if x <= 15 else 2)
    
    # Cria o agrupador 'Mês_Quinzena'
    df_analise['Mês_Quinzena'] = df_analise['Mês'].astype(str) + '_' + df_analise['Quinzena'].astype(str)
    
    # Lista das colunas que queremos analisar
    colunas_bolas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']
    
    # Dicionário para armazenar o resultado final de cada bola
    resultado_por_bola = {}
    
    for bola in colunas_bolas:
        # 2. Agrupamento e Contagem: Conta a frequência de cada número
        # O agrupamento agora é feito por Mês_Quinzena e pelo valor da Bola
        frequencia_tripla = df_analise.groupby(['Mês_Quinzena', bola]).size().reset_index(name='Frequência')

        # 3. Seleção do Top 1: Encontra o número mais frequente em cada Mês-Quinzena
        idx_max = frequencia_tripla.groupby('Mês_Quinzena')['Frequência'].idxmax()

        # Filtramos o DataFrame para manter apenas as linhas com as maiores frequências
        top_numero_por_quinzena = frequencia_tripla.loc[idx_max].copy()

        # Renomeia a coluna da bola para algo mais descritivo
        top_numero_por_quinzena.rename(columns={bola: 'Número Mais Frequente'}, inplace=True)
        
        # Armazena o resultado no dicionário
        resultado_por_bola[bola] = top_numero_por_quinzena.sort_values(by='Mês_Quinzena')

        # Opcional: Imprime o resultado para a bola atual
        print(f"--- Resultados para {bola} (Agrupado por Mês e Quinzena) ---")
        # Usamos apenas as colunas de interesse para exibição
        display(top_numero_por_quinzena[['Mês_Quinzena', 'Número Mais Frequente', 'Frequência']])
        
        print("\n")
        
    print("A análise está concluída. Os resultados estão armazenados no dicionário 'resultado_por_bola'.")
    return resultado_por_bola # Retorna o dicionário com os resultados

In [18]:
df_resultado = Funcao_forma_quinzenal(df_pesquisa)

--- Resultados para Bola1 (Agrupado por Mês e Quinzena) ---


/tmp/ipykernel_21950/2038804472.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_analise['Mês'] = df_analise['Data do Sorteio'].dt.month
/tmp/ipykernel_21950/2038804472.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_analise['Quinzena'] = df_analise['Data do Sorteio'].dt.day.apply(lambda x: 1 if x <= 15 else 2)
/tmp/ipykernel_21950/2038804472.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inste

,Mês_Quinzena,Número Mais Frequente,Frequência
1,10_1,2,17
25,10_2,3,16
54,11_1,4,13
80,11_2,5,15
101,12_1,1,17
126,12_2,1,12
152,1_1,1,14
177,1_2,1,15
205,2_1,2,13
229,2_2,1,13




--- Resultados para Bola2 (Agrupado por Mês e Quinzena) ---


,Mês_Quinzena,Número Mais Frequente,Frequência
8,10_1,11,10
40,10_2,11,14
89,11_1,24,9
109,11_2,8,8
147,12_1,10,9
177,12_2,6,5
221,1_1,13,7
252,1_2,6,8
291,2_1,11,10
330,2_2,16,8


...



--- Resultados para Bola5 (Agrupado por Mês e Quinzena) ---


,Mês_Quinzena,Número Mais Frequente,Frequência
23,10_1,46,7
58,10_2,46,9
98,11_1,53,10
120,11_2,40,10
153,12_1,40,9
190,12_2,45,9
229,1_1,53,8
263,1_2,52,10
285,2_1,37,8
330,2_2,51,8




--- Resultados para Bola6 (Agrupado por Mês e Quinzena) ---


,Mês_Quinzena,Número Mais Frequente,Frequência
24,10_1,59,15
40,10_2,49,13
74,11_1,56,15
105,11_2,60,16
130,12_1,60,17
152,12_2,55,8
179,1_1,58,12
207,1_2,58,12
230,2_1,56,13
251,2_2,58,15




A análise está concluída. Os resultados estão armazenados no dicionário 'resultado_por_bola'.


In [19]:
data_formatada = data_atual.strftime('%d/%m/%Y')
print(f'resultado mensal de forma quinzenal, analisando de forma quinzenal, nos últimos {dias_desejados} dias, a partir do dia de hoje:{data_formatada}')

df_resultado = Funcao_forma_quinzenal(df_ultimos_selecionados_dias)

resultado mensal de forma quinzenal, analisando de forma quinzenal, nos últimos 45 dias, a partir do dia de hoje:13/12/2025
--- Resultados para Bola1 (Agrupado por Mês e Quinzena) ---


,Mês_Quinzena,Número Mais Frequente,Frequência
0,10_2,9,1
1,11_1,4,1
8,11_2,8,2
11,12_1,4,3




--- Resultados para Bola2 (Agrupado por Mês e Quinzena) ---


,Mês_Quinzena,Número Mais Frequente,Frequência
0,10_2,17,1
1,11_1,7,1
10,11_2,30,2
11,12_1,6,1




--- Resultados para Bola3 (Agrupado por Mês e Quinzena) ---


...



--- Resultados para Bola5 (Agrupado por Mês e Quinzena) ---


,Mês_Quinzena,Número Mais Frequente,Frequência
0,10_2,33,1
5,11_1,44,2
6,11_2,27,1
12,12_1,49,2




--- Resultados para Bola6 (Agrupado por Mês e Quinzena) ---


,Mês_Quinzena,Número Mais Frequente,Frequência
0,10_2,59,1
1,11_1,32,1
10,11_2,60,2
12,12_1,54,2




A análise está concluída. Os resultados estão armazenados no dicionário 'resultado_por_bola'.


In [20]:
df_pesquisa.tail()

,Data do Sorteio,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6,Mês,Ano,Mês-Ano,Quinzena,Mês_Quinzena
2945,2025-12-02,4,13,17,21,49,54,12,2025,2025-12,1,12_1
2946,2025-12-04,4,10,15,37,39,44,12,2025,2025-12,1,12_1
2947,2025-12-06,6,24,37,52,53,58,12,2025,2025-12,1,12_1
2948,2025-12-09,4,6,11,38,49,54,12,2025,2025-12,1,12_1
2949,2025-12-11,21,23,42,49,50,60,12,2025,2025-12,1,12_1


## anaconda ambiente orange3

### forecast pelo prophet

In [24]:
#!pip install prophet

In [25]:
import prophet
print(prophet.__version__)

/home/fabiene/anaconda3/envs/orange3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


1.2.1


In [26]:
from prophet import Prophet


### 1️⃣ Código para prever 35 dias para cada bola

In [28]:

resultados = []

bolas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']

for bola in bolas:
    
    # Preparar dados no formato do Prophet
    df_prophet = df_pesquisa[['Data do Sorteio', bola]].rename(
        columns={'Data do Sorteio': 'ds', bola: 'y'}
    )
    
    # Criar e treinar modelo
    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True
    )
    
    model.fit(df_prophet)
    
    # Criar datas futuras (35 dias)
    future = model.make_future_dataframe(periods=35)
    
    forecast = model.predict(future)
    
    # Selecionar apenas previsão futura
    previsao_futura = forecast[['ds', 'yhat']].tail(35)
    previsao_futura['Bola'] = bola
    
    resultados.append(previsao_futura)

# Unir todas as bolas
previsoes = pd.concat(resultados)

previsoes.head()

16:27:36 - cmdstanpy - INFO - Chain [1] start processing
16:27:36 - cmdstanpy - INFO - Chain [1] done processing
16:27:36 - cmdstanpy - INFO - Chain [1] start processing
16:27:37 - cmdstanpy - INFO - Chain [1] done processing
16:27:37 - cmdstanpy - INFO - Chain [1] start processing
16:27:37 - cmdstanpy - INFO - Chain [1] done processing
16:27:37 - cmdstanpy - INFO - Chain [1] start processing
16:27:37 - cmdstanpy - INFO - Chain [1] done processing
16:27:38 - cmdstanpy - INFO - Chain [1] start processing
16:27:38 - cmdstanpy - INFO - Chain [1] done processing
16:27:38 - cmdstanpy - INFO - Chain [1] start processing
16:27:38 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,Bola
2950,2025-12-12,9.317227,Bola1
2951,2025-12-13,9.543757,Bola1
2952,2025-12-14,8.970329,Bola1
2953,2025-12-15,8.631693,Bola1
2954,2025-12-16,9.250559,Bola1


## 2️⃣ Formatar no formato “cada dia com as 6 bolas”

In [29]:
previsoes_final = previsoes.pivot(
    index='ds',
    columns='Bola',
    values='yhat'
).reset_index()

# Arredondar e limitar ao intervalo típico (1 a 60, por exemplo)
for bola in bolas:
    previsoes_final[bola] = previsoes_final[bola].round().clip(1, 60)

previsoes_final.head()

Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,9.0,17.0,24.0,31.0,40.0,51.0
1,2025-12-13,10.0,19.0,26.0,35.0,43.0,52.0
2,2025-12-14,9.0,18.0,24.0,35.0,42.0,51.0
3,2025-12-15,9.0,21.0,27.0,33.0,40.0,51.0
4,2025-12-16,9.0,20.0,26.0,36.0,44.0,52.0


Observação importante 

✔️ O Prophet não aprende regras de sorteio

✔️ Ele apenas extrapola médias e sazonalidades

❌ Não aumenta chance de acerto



In [31]:
previsoes_final.head(45)

Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,9.0,17.0,24.0,31.0,40.0,51.0
1,2025-12-13,10.0,19.0,26.0,35.0,43.0,52.0
2,2025-12-14,9.0,18.0,24.0,35.0,42.0,51.0
3,2025-12-15,9.0,21.0,27.0,33.0,40.0,51.0
4,2025-12-16,9.0,20.0,26.0,36.0,44.0,52.0
5,2025-12-17,9.0,19.0,26.0,35.0,43.0,51.0
6,2025-12-18,10.0,19.0,28.0,36.0,43.0,52.0
7,2025-12-19,9.0,18.0,25.0,32.0,41.0,51.0
8,2025-12-20,9.0,20.0,27.0,36.0,44.0,52.0
9,2025-12-21,9.0,19.0,26.0,36.0,43.0,51.0


# previsão para os últimos 30 dias

🔹 1️⃣ Preparação dos dados (últimos 30 sorteios)

In [34]:

# Usar apenas os últimos 30 sorteios
df_30 = df_pesquisa.sort_values('Data do Sorteio').tail(30).reset_index(drop=True)

In [35]:
df_30

,Data do Sorteio,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6,Mês,Ano,Mês-Ano,Quinzena,Mês_Quinzena
0,2025-09-30,9,12,14,16,26,36,9,2025,2025-09,2,9_2
1,2025-10-02,4,23,30,39,40,41,10,2025,2025-10,1,10_1
2,2025-10-04,18,27,32,39,55,56,10,2025,2025-10,1,10_1
3,2025-10-07,10,19,30,40,48,54,10,2025,2025-10,1,10_1
4,2025-10-09,7,9,12,13,24,27,10,2025,2025-10,1,10_1
5,2025-10-11,3,4,14,35,45,49,10,2025,2025-10,1,10_1
6,2025-10-14,11,27,34,55,56,58,10,2025,2025-10,1,10_1
7,2025-10-16,14,24,29,32,46,48,10,2025,2025-10,2,10_2
8,2025-10-18,3,7,8,34,35,51,10,2025,2025-10,2,10_2
9,2025-10-21,1,11,13,14,36,45,10,2025,2025-10,2,10_2


🔹 2️⃣ Previsão com intervalo de confiança para cada bola

In [36]:
bolas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']
resultados = []

for bola in bolas:
    
    df_prophet = df_30[['Data do Sorteio', bola]].rename(
        columns={'Data do Sorteio': 'ds', bola: 'y'}
    )
    
    model = Prophet(
        weekly_seasonality=True,
        yearly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95  # 95% de confiança
    )
    
    model.fit(df_prophet)
    
    future = model.make_future_dataframe(periods=35)
    forecast = model.predict(future)
    
    futuro = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(35)
    futuro['Bola'] = bola
    
    resultados.append(futuro)

previsoes = pd.concat(resultados)


16:33:42 - cmdstanpy - INFO - Chain [1] start processing
16:33:42 - cmdstanpy - INFO - Chain [1] done processing
16:33:42 - cmdstanpy - INFO - Chain [1] start processing
16:33:42 - cmdstanpy - INFO - Chain [1] done processing
16:33:42 - cmdstanpy - INFO - Chain [1] start processing
16:33:42 - cmdstanpy - INFO - Chain [1] done processing
16:33:42 - cmdstanpy - INFO - Chain [1] start processing
16:33:42 - cmdstanpy - INFO - Chain [1] done processing
16:33:42 - cmdstanpy - INFO - Chain [1] start processing
16:33:42 - cmdstanpy - INFO - Chain [1] done processing
16:33:42 - cmdstanpy - INFO - Chain [1] start processing
16:33:42 - cmdstanpy - INFO - Chain [1] done processing


In [37]:
df_prophet.tail()

,ds,y
25,2025-12-02,54
26,2025-12-04,44
27,2025-12-06,58
28,2025-12-09,54
29,2025-12-11,60


In [39]:
previsoes.tail()

,ds,yhat,yhat_lower,yhat_upper,Bola
60,2026-01-11,-5.189035,-19.708043,11.567725,Bola6
61,2026-01-12,-5.062658,-21.487452,10.986939,Bola6
62,2026-01-13,58.216343,42.719768,74.734792,Bola6
63,2026-01-14,-4.809901,-19.651014,11.644236,Bola6
64,2026-01-15,58.258080,43.316008,74.170332,Bola6


🔹 3️⃣ Organizar no formato “1 dia → 6 bolas”

In [41]:
previsao_central = previsoes.pivot(
    index='ds',
    columns='Bola',
    values='yhat'
).reset_index()

for bola in bolas:
    previsao_central[bola] = (
        previsao_central[bola]
        .round()
        .clip(1, 60)
        .astype(int)
    )

previsao_central.head()


Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,7,8,10,15,24,57
1,2025-12-13,8,16,24,39,46,58
2,2025-12-14,1,1,1,1,1,1
3,2025-12-15,1,1,1,1,1,1
4,2025-12-16,8,19,24,35,46,55
5,2025-12-17,1,1,1,1,1,1
6,2025-12-18,10,18,27,38,44,55
7,2025-12-19,7,8,10,16,24,57
8,2025-12-20,8,16,24,39,47,58
9,2025-12-21,1,1,1,1,1,1


In [51]:
def remover_linhas_com_repeticao(df, colunas_bolas, data):
    """
    Identifica e remove linhas de um DataFrame onde há repetição de números
    nas colunas especificadas, e armazena as datas das linhas removidas.

    Args:
        df (pd.DataFrame): O DataFrame a ser analisado.
        colunas_bolas (list): Lista de nomes das colunas de números.

    Returns:
        tuple: (df_limpo, lista_datas_removidas)
    """

    # 1. Identificar as linhas com números repetidos
    # A função apply(lambda x: x.nunique() != len(x), axis=1) faz o seguinte:
    # - x.nunique(): Conta quantos valores únicos há na linha (nas colunas de bolas).
    # - len(x): É o total de colunas de bolas (que é 6 no seu caso).
    # - x.nunique() != len(x): Retorna True se o número de únicos for diferente do total
    #   de colunas (ou seja, houve repetição).
    
    linhas_com_repeticao = df[colunas_bolas].apply(
        lambda x: x.nunique() != len(x), 
        axis=1
    )

    # 2. Separar o DataFrame
    # DataFrame com as linhas que TÊM repetição (para extrair as datas)
    df_removido = df[linhas_com_repeticao].copy()
    
    # DataFrame limpo (linhas que NÃO TÊM repetição)
    df_limpo = df[~linhas_com_repeticao].copy() # O '~' inverte a seleção (NOT)

    # 3. Extrair a lista de datas das linhas removidas
    # Assumindo que a coluna de data se chama 'Data do Sorteio'
    lista_datas_removidas = df_removido[f'{data}'].tolist()

    return df_limpo, lista_datas_removidas

# --- Exemplo de Uso ---

# Assumindo que seu DataFrame se chama 'df_pesquisa'
colunas = ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']

# Chamando a função
previsao_central_tratada, datas_removidas = remover_linhas_com_repeticao(previsao_central, colunas,'ds')

print(f"Total de linhas removidas devido à repetição: {len(datas_removidas)}")
print("--- Datas Removidas ---")
print(datas_removidas)


Total de linhas removidas devido à repetição: 17
--- Datas Removidas ---
[Timestamp('2025-12-14 00:00:00'), Timestamp('2025-12-15 00:00:00'), Timestamp('2025-12-17 00:00:00'), Timestamp('2025-12-21 00:00:00'), Timestamp('2025-12-22 00:00:00'), Timestamp('2025-12-24 00:00:00'), Timestamp('2025-12-28 00:00:00'), Timestamp('2025-12-29 00:00:00'), Timestamp('2025-12-31 00:00:00'), Timestamp('2026-01-02 00:00:00'), Timestamp('2026-01-04 00:00:00'), Timestamp('2026-01-05 00:00:00'), Timestamp('2026-01-07 00:00:00'), Timestamp('2026-01-09 00:00:00'), Timestamp('2026-01-11 00:00:00'), Timestamp('2026-01-12 00:00:00'), Timestamp('2026-01-14 00:00:00')]


🔹 4️⃣ Intervalo de confiança (opcional, mas útil)

In [52]:
print('🎯 previsao_central → valor “mais provável”')

previsao_central_tratada.head(45)


🎯 previsao_central → valor “mais provável”


Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,7,8,10,15,24,57
1,2025-12-13,8,16,24,39,46,58
4,2025-12-16,8,19,24,35,46,55
6,2025-12-18,10,18,27,38,44,55
7,2025-12-19,7,8,10,16,24,57
8,2025-12-20,8,16,24,39,47,58
11,2025-12-23,9,19,24,36,46,56
13,2025-12-25,10,18,27,38,44,56
14,2025-12-26,7,8,11,17,25,58
15,2025-12-27,8,16,25,40,47,59


limite inferior



In [42]:
previsao_min = previsoes.pivot(
    index='ds',
    columns='Bola',
    values='yhat_lower'
).reset_index()

for bola in bolas:
    previsao_min[bola] = previsao_min[bola].round().clip(1, 60).astype(int)


In [48]:
print("🔻 previsao_min → cenário conservador")
previsao_min

🔻 previsao_min → cenário conservador


Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,1,1,1,1,8,40
1,2025-12-13,1,1,5,16,30,42
2,2025-12-14,1,1,1,1,1,1
3,2025-12-15,1,1,1,1,1,1
4,2025-12-16,1,3,3,14,30,39
5,2025-12-17,1,1,1,1,1,1
6,2025-12-18,1,1,8,15,27,40
7,2025-12-19,1,1,1,1,9,41
8,2025-12-20,1,1,6,17,31,43
9,2025-12-21,1,1,1,1,1,1


limite superior

In [57]:
previsao_min.loc[previsao_min['ds'].isin(datas_removidas)].reset_index(drop=True)

Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-14,1,1,1,1,1,1
1,2025-12-15,1,1,1,1,1,1
2,2025-12-17,1,1,1,1,1,1
3,2025-12-21,1,1,1,1,1,1
4,2025-12-22,1,1,1,1,1,1
5,2025-12-24,1,1,1,1,1,1
6,2025-12-28,1,1,1,1,1,1
7,2025-12-29,1,1,1,1,1,1
8,2025-12-31,1,1,1,1,1,1
9,2026-01-02,1,1,1,1,11,44


In [59]:
previsao_min_tratada, datas_removidas = remover_linhas_com_repeticao(previsao_min, colunas,'ds')

print(f"Total de linhas removidas devido à repetição: {len(datas_removidas)}")
print("--- Datas Removidas ---")
print(datas_removidas)

Total de linhas removidas devido à repetição: 27
--- Datas Removidas ---
[Timestamp('2025-12-12 00:00:00'), Timestamp('2025-12-13 00:00:00'), Timestamp('2025-12-14 00:00:00'), Timestamp('2025-12-15 00:00:00'), Timestamp('2025-12-16 00:00:00'), Timestamp('2025-12-17 00:00:00'), Timestamp('2025-12-18 00:00:00'), Timestamp('2025-12-19 00:00:00'), Timestamp('2025-12-20 00:00:00'), Timestamp('2025-12-21 00:00:00'), Timestamp('2025-12-22 00:00:00'), Timestamp('2025-12-24 00:00:00'), Timestamp('2025-12-26 00:00:00'), Timestamp('2025-12-27 00:00:00'), Timestamp('2025-12-28 00:00:00'), Timestamp('2025-12-29 00:00:00'), Timestamp('2025-12-31 00:00:00'), Timestamp('2026-01-02 00:00:00'), Timestamp('2026-01-04 00:00:00'), Timestamp('2026-01-05 00:00:00'), Timestamp('2026-01-07 00:00:00'), Timestamp('2026-01-09 00:00:00'), Timestamp('2026-01-10 00:00:00'), Timestamp('2026-01-11 00:00:00'), Timestamp('2026-01-12 00:00:00'), Timestamp('2026-01-14 00:00:00'), Timestamp('2026-01-15 00:00:00')]


In [60]:
previsao_min_tratada

Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
11,2025-12-23,1,4,6,13,30,40
13,2025-12-25,1,2,9,18,29,40
18,2025-12-30,1,3,6,15,31,41
20,2026-01-01,1,2,8,16,29,41
22,2026-01-03,1,2,5,18,32,46
25,2026-01-06,1,4,6,14,32,42
27,2026-01-08,1,2,8,18,29,40
32,2026-01-13,1,3,7,17,32,43


In [62]:
previsao_min_tratada['ds'].nunique()

8

In [43]:
previsao_max = previsoes.pivot(
    index='ds',
    columns='Bola',
    values='yhat_upper'
).reset_index()

for bola in bolas:
    previsao_max[bola] = previsao_max[bola].round().clip(1, 60).astype(int)


In [49]:
print(f"🔺 previsao_max → cenário otimista") # resultado estanho
previsao_max

🔺 previsao_max → cenário otimista


Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,18,24,29,38,40,60
1,2025-12-13,18,30,43,60,60,60
2,2025-12-14,8,11,17,18,6,8
3,2025-12-15,9,10,14,18,7,7
4,2025-12-16,19,35,43,57,60,60
5,2025-12-17,8,10,14,19,8,9
6,2025-12-18,20,34,47,59,60,60
7,2025-12-19,18,24,30,38,41,60
8,2025-12-20,19,32,43,60,60,60
9,2025-12-21,8,10,16,19,9,7


In [63]:
previsao_max_tratada, datas_removidas = remover_linhas_com_repeticao(previsao_max, colunas,'ds')

print(f"Total de linhas removidas devido à repetição: {len(datas_removidas)}")
print("--- Datas Removidas ---")
print(datas_removidas)

Total de linhas removidas devido à repetição: 28
--- Datas Removidas ---
[Timestamp('2025-12-13 00:00:00'), Timestamp('2025-12-14 00:00:00'), Timestamp('2025-12-15 00:00:00'), Timestamp('2025-12-16 00:00:00'), Timestamp('2025-12-17 00:00:00'), Timestamp('2025-12-18 00:00:00'), Timestamp('2025-12-20 00:00:00'), Timestamp('2025-12-22 00:00:00'), Timestamp('2025-12-23 00:00:00'), Timestamp('2025-12-24 00:00:00'), Timestamp('2025-12-25 00:00:00'), Timestamp('2025-12-27 00:00:00'), Timestamp('2025-12-28 00:00:00'), Timestamp('2025-12-29 00:00:00'), Timestamp('2025-12-30 00:00:00'), Timestamp('2025-12-31 00:00:00'), Timestamp('2026-01-01 00:00:00'), Timestamp('2026-01-03 00:00:00'), Timestamp('2026-01-04 00:00:00'), Timestamp('2026-01-05 00:00:00'), Timestamp('2026-01-06 00:00:00'), Timestamp('2026-01-07 00:00:00'), Timestamp('2026-01-08 00:00:00'), Timestamp('2026-01-10 00:00:00'), Timestamp('2026-01-11 00:00:00'), Timestamp('2026-01-13 00:00:00'), Timestamp('2026-01-14 00:00:00'), Timestam

In [64]:
previsao_max_tratada

Bola,ds,Bola1,Bola2,Bola3,Bola4,Bola5,Bola6
0,2025-12-12,18,24,29,38,40,60
7,2025-12-19,18,24,30,38,41,60
9,2025-12-21,8,10,16,19,9,7
14,2025-12-26,18,24,30,40,41,60
21,2026-01-02,17,24,30,39,42,60
28,2026-01-09,18,23,32,38,42,60
31,2026-01-12,9,8,18,20,10,11
